In [ ]:
# Cell 1: imports / paths / device / test patch dataset

import os
from pathlib import Path
import random

import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms, datasets

from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Paths
REPO_ROOT    = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
RAW_ROOT     = REPO_ROOT / "data" / "raw" / "pcb defect.v1i.yolov5pytorch"
SW_PATCH_ROOT = REPO_ROOT / "data" / "patches_sw"
RESULTS_DIR  = REPO_ROOT / "results"

print("REPO_ROOT :", REPO_ROOT)
print("RAW_ROOT  :", RAW_ROOT)
print("SW_ROOT   :", SW_PATCH_ROOT)
print("RESULTS   :", RESULTS_DIR)

# Device 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

#  Constants 
PATCH_SIZE = 128

CLASSES = [
    "background",
    "missing_hole",
    "mouse_bite",
    "open_circuit",
    "short",
    "spur",
    "spurious_copper",
]

# Inference transform (same stats as training)
inference_transform = transforms.Compose([
    transforms.Resize((PATCH_SIZE, PATCH_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],   # ImageNet mean
        std=[0.229, 0.224, 0.225],    # ImageNet std
    ),
])

# Test patch dataset (sliding-window patches)
test_patch_dir = SW_PATCH_ROOT / "test"
test_dataset = datasets.ImageFolder(
    root=test_patch_dir,
    transform=inference_transform
)
print("Test patch dataset size:", len(test_dataset))
print("Classes in ImageFolder:", test_dataset.classes)


In [ ]:
# Cell 2: build MobileNetV2 and load trained weights

num_classes = len(CLASSES)

model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
model = model.to(device)

CKPT_NAME = "mobilenetv2_sw_best.pth" 
ckpt_path = RESULTS_DIR / CKPT_NAME

print("Loading checkpoint:", ckpt_path)
state = torch.load(ckpt_path, map_location=device)
model.load_state_dict(state)
model.eval()

print("Model loaded and in eval mode.")


In [ ]:
# Cell 3: helper functions for patch prediction

def predict_patch(pil_img):
    """Run a single PIL patch through the model, return (pred_idx, probs)."""
    model.eval()
    with torch.no_grad(), torch.amp.autocast(
        device_type=device.type,
        dtype=torch.float16,
        enabled=(device.type == "cuda")
    ):
        x = inference_transform(pil_img).unsqueeze(0).to(device)
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
    pred_idx = int(np.argmax(probs))
    return pred_idx, probs


def show_patch_with_prediction(pil_img, true_label=None, ax=None):
    pred_idx, probs = predict_patch(pil_img)
    pred_label = CLASSES[pred_idx]

    if ax is None:
        fig, ax = plt.subplots(figsize=(3, 3))

    ax.imshow(pil_img)
    ax.axis("off")
    title = f"Pred: {pred_label}"
    if true_label is not None:
        title = f"True: {true_label}\nPred: {pred_label}"
    ax.set_title(title, fontsize=9)

    print("Top class probabilities:")
    for cls, p in sorted(zip(CLASSES, probs), key=lambda x: x[1], reverse=True):
        print(f"  {cls:15s} : {p:0.3f}")
    print()


In [ ]:
# Cell 4: run model on two random test patches (side-by-side)

indices = random.sample(range(len(test_dataset)), 2)

fig, axes = plt.subplots(1, 2, figsize=(6, 3))

for ax, idx in zip(axes, indices):
    img_path, label_idx = test_dataset.samples[idx]  # path + class index
    patch_pil = Image.open(img_path).convert("RGB")
    true_label = CLASSES[label_idx]

    print(f"Patch index: {idx}")
    print("File:", img_path)
    show_patch_with_prediction(patch_pil, true_label=true_label, ax=ax)

plt.tight_layout()
plt.show()


In [ ]:
# Cell 5: visualize the patch grid on a full PCB image (no predictions yet)

def visualize_patch_grid(img_pil, window_size=PATCH_SIZE, stride=None):
    if stride is None:
        stride = window_size // 2   

    W, H = img_pil.size
    plt.figure(figsize=(6, 6))
    plt.imshow(img_pil)
    ax = plt.gca()

    for y in range(0, H - window_size + 1, stride):
        for x in range(0, W - window_size + 1, stride):
            rect = plt.Rectangle(
                (x, y),
                window_size,
                window_size,
                fill=False,
                linewidth=0.8
            )
            ax.add_patch(rect)

    plt.axis("off")
    plt.title(f"Patch grid (size={window_size}, stride={stride})")
    plt.show()


# Use a sample train image
full_img_dir = RAW_ROOT / "train" / "images"
img_path = sorted(full_img_dir.glob("*.jpg"))[0]
print("Using image for patch grid:", img_path)

full_img = Image.open(img_path).convert("RGB")
print("Image size:", full_img.size)

visualize_patch_grid(full_img, window_size=PATCH_SIZE, stride=PATCH_SIZE // 2)


In [ ]:
# Cell 6: IoU, NMS, sliding-window detection, visualization

def iou(box1, box2):
    # boxes: [x1, y1, x2, y2]
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_w = max(0.0, x2 - x1)
    inter_h = max(0.0, y2 - y1)
    inter = inter_w * inter_h
    if inter == 0:
        return 0.0

    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union


def nms(detections, iou_thresh=0.3):
    """
    detections: list of (x1, y1, x2, y2, cls_idx, prob)
    returns: pruned list, using IoU for suppression
    """
    if not detections:
        return []

    dets = sorted(detections, key=lambda d: d[5], reverse=True)
    kept = []

    while dets:
        best = dets.pop(0)
        kept.append(best)
        remaining = []
        for d in dets:
            if iou(best[:4], d[:4]) < iou_thresh:
                remaining.append(d)
        dets = remaining

    return kept


def run_sliding_window_on_image(img_pil,
                                window_size=PATCH_SIZE,
                                stride=None,
                                prob_thresh=0.6):
    """
    Slide a window over the image and classify each patch.
    Returns list of detections: (x1, y1, x2, y2, cls_idx, prob)
    """
    if stride is None:
        stride = window_size // 2  # 50% overlap

    W, H = img_pil.size
    detections = []

    for y in range(0, H - window_size + 1, stride):
        for x in range(0, W - window_size + 1, stride):
            patch = img_pil.crop((x, y, x + window_size, y + window_size))
            pred_idx, probs = predict_patch(patch)

            prob = float(probs[pred_idx])
            if CLASSES[pred_idx] == "background":
                continue
            if prob < prob_thresh:
                continue

            detections.append((x, y, x + window_size, y + window_size, pred_idx, prob))

    return detections


def visualize_detections(full_img_pil, detections, title="Detections"):
    plt.figure(figsize=(6, 6))
    plt.imshow(full_img_pil)
    ax = plt.gca()
    for (x1, y1, x2, y2, cls_idx, prob) in detections:
        w = x2 - x1
        h = y2 - y1
        rect = plt.Rectangle(
            (x1, y1), w, h,
            fill=False,
            linewidth=2
        )
        ax.add_patch(rect)
        label = f"{CLASSES[cls_idx]}: {prob:.2f}"
        ax.text(
            x1, y1 - 3, label,
            fontsize=8,
            color="yellow",
            bbox=dict(facecolor="black", alpha=0.5, pad=1),
        )
    plt.axis("off")
    plt.title(title)
    plt.show()


In [ ]:
# Cell 7: sliding-window inference + NMS on a full image

full_img_dir = RAW_ROOT / "train" / "images"
img_path = sorted(full_img_dir.glob("*.jpg"))[100]  
print("Using image for sliding window:", img_path)

full_img = Image.open(img_path).convert("RGB")
print("Image size:", full_img.size)

# 1) Sliding-window detections (per-patch CNN)
raw_dets = run_sliding_window_on_image(
    full_img,
    window_size=PATCH_SIZE,
    stride=PATCH_SIZE // 2,
    prob_thresh=0.6,
)
print("Raw detections (before NMS):", len(raw_dets))

# 2) IoU-based NMS
dets_nms = nms(raw_dets, iou_thresh=0.3)
print("Detections after NMS:", len(dets_nms))

visualize_detections(
    full_img,
    dets_nms,
    title="Sliding-window detections (IoU-based NMS)",
)


In [ ]:
# Cell 8 compare detections to YOLO boxes using IoU

YOLO_CLASSES = CLASSES[1:]  # YOLO has 6 defect classes, no background


def read_yolo_labels(label_path, img_w, img_h):
    """
    Read YOLO txt -> list of (cls_name, x1, y1, x2, y2) in pixels.
    """
    boxes = []
    if not label_path.exists():
        return boxes

    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls_idx = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:])

            cx *= img_w
            cy *= img_h
            bw *= img_w
            bh *= img_h

            x1 = cx - bw / 2
            y1 = cy - bh / 2
            x2 = cx + bw / 2
            y2 = cy + bh / 2

            x1 = max(0, min(img_w - 1, x1))
            y1 = max(0, min(img_h - 1, y1))
            x2 = max(0, min(img_w - 1, x2))
            y2 = max(0, min(img_h - 1, y2))

            cls_name = YOLO_CLASSES[cls_idx]
            boxes.append((cls_name, x1, y1, x2, y2))
    return boxes


def evaluate_detections_iou(img_path, detections):
    img = Image.open(img_path).convert("RGB")
    W, H = img.size

    label_path = img_path.parent.parent / "labels" / (img_path.stem + ".txt")
    gt_boxes = read_yolo_labels(label_path, W, H)

    if not gt_boxes:
        print("No YOLO label file found or no boxes.")
        return

    print("\nDetection IoUs vs ground truth boxes:")
    ious = []

    for (x1, y1, x2, y2, cls_idx, prob) in detections:
        pred_cls = CLASSES[cls_idx]
        best_iou = 0.0
        for (gt_cls, gx1, gy1, gx2, gy2) in gt_boxes:
            if gt_cls != pred_cls:
                continue
            best_iou = max(best_iou, iou((x1, y1, x2, y2), (gx1, gy1, gx2, gy2)))

        ious.append(best_iou)
        print(f"{pred_cls:15s} prob={prob:.2f}  best IoU={best_iou:.3f}")

    if ious:
        print(f"\nMean IoU over detections: {np.mean(ious):.3f}")
    else:
        print("No matching GT boxes for any detections.")


# Run IoU eval on the same image/detections from Cell 7
evaluate_detections_iou(img_path, dets_nms)
